# LAB3 — Bài 2: MNIST với train/validation

**Sinh viên:** Trần Chí Nguyên — **MSSV:** 6451071052.

> Mỗi câu lập trình in họ tên và MSSV theo template. Chạy lần lượt từ trên xuống.

In [1]:
%matplotlib inline
HO_TEN = 'Trần Chí Nguyên'
MSSV = '6451071052'
def in_danh_tinh():
    print(f'{HO_TEN} - {MSSV}')


## Câu a–d. Đọc, khảo sát, chuẩn hóa và chia train/validation

Đọc MNIST từ Keras, in shape, chuẩn hóa pixel về [0,1] và chia train gốc 8:2.

In [2]:
in_danh_tinh()
import numpy as np
from tensorflow.keras.datasets import mnist
from sklearn.model_selection import train_test_split
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()
print('Train:', train_images.shape, train_labels.shape)
print('Test:', test_images.shape, test_labels.shape)
train_images = train_images.astype('float32') / 255.0
test_images = test_images.astype('float32') / 255.0
X_train, X_val, y_train, y_val = train_test_split(
    train_images, train_labels, test_size=0.2, random_state=42, stratify=train_labels
)
print('Train:', X_train.shape, y_train.shape, '| Validation:', X_val.shape, y_val.shape)

Trần Chí Nguyên - 6451071052


       0/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0s/step

   16384/11490434 ━━━━━━━━━━━━━━━━━━━━ 40s 3us/step

   49152/11490434 ━━━━━━━━━━━━━━━━━━━━ 51s 4us/step

   81920/11490434 ━━━━━━━━━━━━━━━━━━━━ 59s 5us/step

  114688/11490434 ━━━━━━━━━━━━━━━━━━━━ 47s 4us/step

... (25 outputs left out)

11075584/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Train: (60000, 28, 28) (60000,)
Test: (10000, 28, 28) (10000,)


Train: (48000, 28, 28) (48000,) | Validation: (12000, 28, 28) (12000,)


## Câu e–g. Xây dựng và huấn luyện mạng nơ-ron

Flatten → Dense(128, ReLU) → Dropout(0.2) → Dense(10, softmax), optimizer Adam, sparse categorical crossentropy, accuracy; 5 epochs, batch_size 32.

In [3]:
in_danh_tinh()
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Flatten, Dense, Dropout
model = Sequential([Flatten(input_shape=(28, 28)), Dense(128, activation='relu'),
                    Dropout(0.2), Dense(10, activation='softmax')])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=5, batch_size=32, verbose=1)

Trần Chí Nguyên - 6451071052


C:\Users\Nguyen\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/5


   1/1500 ━━━━━━━━━━━━━━━━━━━━ 1:00:06 2s/step - accuracy: 0.0000e+00 - loss: 2.5464

  12/1500 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.3177 - loss: 2.0592        

  25/1500 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.4875 - loss: 1.7351

... (998 outputs left out)

1475/1500 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9747 - loss: 0.0802

1485/1500 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9747 - loss: 0.0801

1495/1500 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9747 - loss: 0.0800

1500/1500 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - accuracy: 0.9747 - loss: 0.0798 - val_accuracy: 0.9752 - val_loss: 0.0874


## Câu h. Vẽ accuracy và loss

Vẽ các đường accuracy/loss của train và validation theo epoch.

In [4]:
in_danh_tinh()
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric, val_metric in [(axes[0], 'accuracy', 'val_accuracy'), (axes[1], 'loss', 'val_loss')]:
    ax.plot(history.history[metric], label='train'); ax.plot(history.history[val_metric], label='validation')
    ax.set_xlabel('Epoch'); ax.set_title(metric); ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()

Trần Chí Nguyên - 6451071052


  plt.tight_layout(); plt.show()


## Câu i–l. Dự đoán và đánh giá tập test

Tính accuracy, precision/recall/F1 theo lớp và confusion matrix.

In [5]:
in_danh_tinh()
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
prob = model.predict(test_images, batch_size=256, verbose=0)
y_pred = np.argmax(prob, axis=1)
print('Test accuracy:', accuracy_score(test_labels, y_pred))
print(classification_report(test_labels, y_pred, digits=4, zero_division=0))
sns.heatmap(confusion_matrix(test_labels, y_pred), cmap='Blues', annot=False)
plt.xlabel('Dự đoán'); plt.ylabel('Thực tế'); plt.title('MNIST — Confusion matrix'); plt.show()

Trần Chí Nguyên - 6451071052


Test accuracy: 0.9757
              precision    recall  f1-score   support

           0     0.9720    0.9929    0.9823       980
           1     0.9851    0.9930    0.9890      1135
           2     0.9720    0.9767    0.9744      1032
           3     0.9695    0.9772    0.9734      1010
           4     0.9805    0.9715    0.9760       982
           5     0.9786    0.9720    0.9753       892
           6     0.9801    0.9791    0.9796       958
           7     0.9671    0.9737    0.9704      1028
           8     0.9771    0.9651    0.9711       974
           9     0.9747    0.9534    0.9639      1009

    accuracy                         0.9757     10000
   macro avg     0.9757    0.9755    0.9755     10000
weighted avg     0.9757    0.9757    0.9757     10000



## Câu m. Hiển thị dự đoán năm ảnh đầu

Hiển thị ảnh test, nhãn thật và nhãn dự đoán cho 5 mẫu đầu.

In [6]:
in_danh_tinh()
fig, axes = plt.subplots(1, 5, figsize=(12, 3))
for i, ax in enumerate(axes):
    ax.imshow(np.expand_dims(test_images[i], axis=0)[0], cmap='gray')
    ax.set_title(f'Thật: {test_labels[i]}\nDự đoán: {y_pred[i]}'); ax.axis('off')
plt.tight_layout(); plt.show()

Trần Chí Nguyên - 6451071052


  plt.tight_layout(); plt.show()


## Câu n. Năm mẫu phân lớp sai đầu tiên

Tìm và hiển thị tối đa năm ảnh đầu tiên có dự đoán khác nhãn thật.

In [7]:
in_danh_tinh()
wrong = np.flatnonzero(y_pred != test_labels)[:5]
print('Chỉ số mẫu sai:', wrong.tolist())
if len(wrong) == 0: print('Không có mẫu sai.')
else:
    fig, axes = plt.subplots(1, len(wrong), figsize=(3*len(wrong), 3))
    axes = np.atleast_1d(axes)
    for i, ax in zip(wrong, axes):
        ax.imshow(test_images[i], cmap='gray'); ax.set_title(f'Thật {test_labels[i]} / Dự đoán {y_pred[i]}'); ax.axis('off')
    plt.tight_layout(); plt.show()

Trần Chí Nguyên - 6451071052
Chỉ số mẫu sai: [8, 151, 217, 241, 247]


  plt.tight_layout(); plt.show()
